In [69]:
import pandas as pd
import joblib
import numpy as np
import warnings
warnings.filterwarnings("ignore")
from tqdm import tqdm

In [ ]:
exclusive_rules = pd.read_csv('../Rules/KDD99_exclusive_rules.csv')
X_test = pd.read_csv('Datasets/KDD99/X_test.csv')
y_test = pd.read_csv('Datasets/KDD99/y_test.csv')

model = joblib.load('../Models/KDD99.pkl')
feature_names = X_test.columns

In [75]:
from collections import Counter

# Initialize dictionary to store rule distribution
dist = Counter()

# Precompute exclusive rule counts as a dictionary for faster lookups
exclusive_rule_counts = exclusive_rules.groupby(['feature', 'value']).size().to_dict()

# Process each row in X_test
for _, test_sample in tqdm(X_test.iterrows(), total=len(X_test), desc="Processing Samples"):
    # Count matching rules
    c = sum(exclusive_rule_counts.get((feature, test_sample[feature]), 0) for feature in feature_names)

    # Update distribution count
    dist[c] += 1

dist


Processing Samples:   0%|          | 0/29117 [00:00<?, ?it/s]

Processing Samples: 100%|██████████| 29117/29117 [00:07<00:00, 3715.34it/s]


Counter({2: 10597, 0: 8781, 1: 8265, 3: 1377, 4: 94, 5: 3})

In [212]:
test_sample = X_test.sample(1) # best indice is 27243
print(f'Incident i prédit: {model.predict(test_sample)[0]}')
print('Les observations o:')
test_sample

Incident i prédit: 0
Les observations o:


,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_count,dst_host_srv_count,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate
13575,0,1,22,9,293,169,0,0,0,0,...,10,255,1.0,0.0,0.1,0.05,0.1,0.01,0.0,0.0


In [120]:
# Initialize an empty list to store matching rules
matched_rules = []

# Iterate over features and find matching rules
for feature in feature_names:
    matching_rules = exclusive_rules[
        (exclusive_rules['feature'] == feature) & 
        (exclusive_rules['value'] == test_sample[feature].values[0])  # Corrected: No .values[0] needed
    ]
    matched_rules.append(matching_rules)

# Concatenate all matched rules into a single DataFrame
rules = pd.concat(matched_rules, ignore_index=True) if matched_rules else pd.DataFrame(columns=['feature', 'value', 'rule', 'support', 'confidence', 'lift'])

rules.sort_values('pct', ascending=False)

,feature,is,value,predict,support,pct,ids
0,src_bytes,==,45.00,0,572,0.49,"[121, 143, 549, 563, 616, 766, 823, 842, 872, ..."
3,srv_diff_host_rate,==,0.01,0,324,0.28,"[435, 766, 872, 1072, 1231, 2437, 3087, 3370, ..."
1,dst_bytes,==,78.00,0,20,0.02,"[2437, 26267, 29426, 31164, 32903, 51227, 5196..."
2,srv_count,==,155.00,0,10,0.01,"[3087, 26131, 62435, 62665, 70627, 74125, 7671..."


### Personalisation options

In [116]:
observations = rules[['feature', 'value']].values
prediction = rules['predict'].values[0]

In [213]:
# option 1: One big rule since predic is always the same
rule = f"Si {' OU '.join([f'{f} = {v}' for f, v in observations])} ALORS incident = {prediction}"
print(rule)

Si src_bytes = 45.0 OU dst_bytes = 78.0 OU srv_count = 155.0 OU srv_diff_host_rate = 0.01 ALORS incident = 0


In [118]:
# Option 2: Interest group
interest_observations = ['src_bytes', 'dst_bytes']

# if the intersest group is in the observations, print them first then make a separation
if set(interest_observations).issubset(set(rules['feature'].values)):
    print('Interest group:')
    for f, v in observations:
        if f in interest_observations:
            print(f'{f} = {v}')
    print('-----------------')
    for f, v in observations:
        if f not in interest_observations:
            print(f'{f} = {v}')

Interest group:
src_bytes = 45.0
dst_bytes = 78.0
-----------------
srv_count = 155.0
srv_diff_host_rate = 0.01


In [ ]:
# Option 3: how many examples have the exact same observations
X_train = pd.read_csv('Datasets/KDD99/X_train.csv')
y_train = pd.read_csv('Datasets/KDD99/y_train.csv')

In [215]:
from itertools import combinations

# Convert the column into a list of sets
id_sets = [set(eval(ids)) if isinstance(ids, str) else set(ids) for ids in rules['ids'].values]

# Step 1: Try full intersection
common_ids = set.intersection(*id_sets) if len(id_sets) > 1 else id_sets[0]
best_combination = "All rows"
f_names = list(rules['feature'].values)  # Extract feature names from rules

# Step 2: If no common elements, reduce the combination size dynamically
if not common_ids:
    for size in range(len(id_sets) - 1, 1, -1):  # From (n-1) down to 2
        for comb in combinations(range(len(id_sets)), size):
            current_common = set.intersection(*[id_sets[i] for i in comb])
            if current_common:
                common_ids = current_common
                feature_subset = [f_names[i] for i in comb]  # Convert indices to names
                best_combination = f"combination of {size}/{len(observations)} features: \n{', '.join(map(str, feature_subset))}"
                break
        if common_ids:
            break

print(f"Total common samples: {len(common_ids)}")
print(f"Common IDs: {common_ids}")
print(f"Label: ",y_train.loc[101489].values[0])
print(f"Found in {best_combination}")

Total common samples: 1
Common IDs: {101489}
Label:  0
Found in combination of 3/4 features: 
src_bytes, dst_bytes, srv_diff_host_rate


In [205]:
import pandas as pd

def merge_exclusive_rules(exclusive_rules):
    """
    Merge consecutive values into ranges for the same feature and prediction.
    
    :param exclusive_rules: DataFrame containing ['feature', 'value', 'predict'] columns.
    :return: DataFrame with merged ranges as 'min_value' and 'max_value' columns.
    """
    merged_rules = []
    
    # Sort the rules to group consecutive values
    exclusive_rules = exclusive_rules.sort_values(by=['feature', 'value']).reset_index(drop=True)
    
    current_feature, current_predict = None, None
    start_value, end_value = None, None
    
    for _, row in exclusive_rules.iterrows():
        feature, value, predict = row['feature'], row['value'], row['predict']
        
        # If continuing the same feature & prediction
        if feature == current_feature and predict == current_predict and value == end_value + 1:
            end_value = value  # Expand the range
        else:
            # Save the previous range if exists
            if start_value is not None:
                merged_rules.append({'feature': current_feature, 'min_value': start_value, 'max_value': end_value, 'predict': current_predict})
            
            # Start a new range
            current_feature, current_predict = feature, predict
            start_value, end_value = value, value
    
    # Save the last range
    if start_value is not None:
        merged_rules.append({'feature': current_feature, 'min_value': start_value, 'max_value': end_value, 'predict': current_predict})
    
    return pd.DataFrame(merged_rules)

# Example Usage:
merged_exclusive_rules = merge_exclusive_rules(exclusive_rules)

In [ ]:
# Initialize the DataFrame to store matching rules
context_rules = []

# Iterate through features and find matching rules
for feature in feature_names:
    feature_value = test_sample[feature].values[0]

    matching_rules = merged_exclusive_rules[
        (merged_exclusive_rules['feature'] == feature) &
        (feature_value >= merged_exclusive_rules['min_value']) &
        (feature_value <= merged_exclusive_rules['max_value'])
    ]
    
    if not matching_rules.empty:
        context_rules.append(matching_rules)

# Concatenate all matching rules into a single DataFrame
context_rules = pd.concat(context_rules, ignore_index=True) if context_rules else pd.DataFrame(columns=['feature', 'min_value', 'max_value', 'predict'])
context_rules

,feature,min_value,max_value,predict
0,src_bytes,45.00,50.00,0
1,dst_bytes,78.00,90.00,0
2,srv_count,153.00,155.00,0
3,srv_diff_host_rate,0.01,0.01,0
